#### LangGraph 기본 예제


##### 1) 라이브러리 설치

##### 2) OpenAI 인증키 설정

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

##### 두 개의 AI 에이전트 협력

In [ ]:

from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI
from pprint import pprint

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model='gpt-4o-mini') # 테스트의 경우에는 작은 모델을 사용합니다

from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
)
print(llm.model_name)

# 첫 번째 AI 에이전트: 질문 분석 및 배경 정보 생성
def agent_1(state):
    """사용자의 질문을 분석하고 핵심 키워드와 배경 정보를 추가"""
    query = state["query"]
    
    # 질문에서 핵심 키워드 추출
    keywords = llm.invoke(f"질문: {query}\n이 질문에서 핵심 키워드를 3~5개 추출해 주세요.")
    
    # 질문과 관련된 배경 정보 제공
    background_info = llm.invoke(f"질문: {query}\n이 질문을 이해하는 데 도움이 될 만한 추가 정보를 제공해 주세요.")

    print(f"\n[Agent 1] 원본 질문: {query}")
    print(f"[Agent 1] 핵심 키워드: {keywords}")
    print(f"[Agent 1] 배경 정보: {background_info}\n")

    return {"refined_query": query, "keywords": keywords, "background_info": background_info}

# 두 번째 AI 에이전트: 키워드 및 배경 정보를 활용하여 답변 생성
def agent_2(state):
    """Agent 1이 제공한 정보를 기반으로 보다 정교한 답변 생성"""
    refined_query = state["refined_query"]
    keywords = state["keywords"]
    background_info = state["background_info"]

    # Agent 1이 제공한 정보를 활용하여 최종 답변 생성
    final_response = llm.invoke(
        f"질문: {refined_query}\n"
        f"핵심 키워드: {keywords}\n"
        f"배경 정보: {background_info}\n"
        f"위 정보를 바탕으로 질문에 대해 깊이 있는 답변을 작성해 주세요."
    )

    print(f"[Agent 2] 최종 답변 생성 완료\n")
    
    return {"final_answer": final_response}

# LangGraph Workflow 설정
workflow = StateGraph(dict)  

# 그래프의 시작점 정의
workflow.add_node("agent_1", agent_1)
workflow.add_node("agent_2", agent_2)

# 실행 흐름(Edges) 정의
workflow.set_entry_point("agent_1")  # Agent 1이 먼저 실행됨
workflow.add_edge("agent_1", "agent_2")  # Agent 1 -> Agent 2

# 실행 엔진 빌드
app = workflow.compile()

mermaid_code = app.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

client=<openai.resources.chat.completions.completions.Completions object at 0x00000250C11250D0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000250C1127A10> model_name='solar-pro' temperature=0.5 model_kwargs={} upstage_api_key=SecretStr('**********') upstage_api_base='https://api.upstage.ai/v1'
Mermaid Code:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent_1(agent_1)
	agent_2(agent_2)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent_1;
	agent_1 --> agent_2;
	agent_2 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[Graph 이미지](https://mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNpVkcFugzAMhl8l8i6tBAySltC06mV9hJ02piqFBCJBQCFI66q--9KUsfZk-9fvz3ZygaIrBTCoDO9r9H7Y5jq3x-NguXFh8bnr93O1e-33X0vGmFRmsDcjr4S2x2QxxeW_hicNL-9AocsZ5_MZ1vA7ax6DwnCPJuL2YcqDjh907PWJ6vXCIYeDkKgUko-NRVI1DXuRWMZSBo3SIqyFqmrLkgg_NfjDvD3sel4oe2bxk-G27IQ7yVMqCwjc06kSmOTNIAJohWn5rYZLrhHKwdaiFTkwl07r5JDrq-vruf7ouhaYNaPrNN1Y1X_F2JfcioPi7l_aGW7cjcK8daO2wBLiEcAu8A2MulMoifGabNKErDbrAM7Os8qiLMXZBq8zsqI4uwbw42fGEaUEU4yTlOA4JjS7_gKcdazK)

In [2]:

# 실행 예제
query = "LangGraph는 무엇이며, LangChain과 어떤 차이점이 있나요? 그리고 LangGraph를 사용해야 하는 이유는 무엇인가요?"
state = {"query": query}
result = app.invoke(state)

# 최종 결과 출력
print("\n [AI 최종 답변]:")
pprint(result)
pprint(result["final_answer"].content)




[Agent 1] 원본 질문: LangGraph는 무엇이며, LangChain과 어떤 차이점이 있나요? 그리고 LangGraph를 사용해야 하는 이유는 무엇인가요?
[Agent 1] 핵심 키워드: content='핵심 키워드:  \n1. **LangGraph**  \n2. **LangChain** (차이점)  \n3. **사용 이유**  \n\n추가 키워드 (필요시):  \n4. **상태 관리** (LangGraph의 주요 기능)  \n5. **멀티모달 워크플로우** (활용 사례)  \n\n이유:  \n- "LangGraph"와 "LangChain"은 비교 대상인 핵심 기술 이름입니다.  \n- "차이점"과 "사용 이유"는 질문의 목적을 명확히 반영합니다.  \n- 부가적으로 LangGraph의 특징인 **상태 관리**나 **워크플로우 설계**를 포함할 수 있으나, 질문의 초점은 기본 개념에 집중되어 있으므로 3~5개 범위 내에서 선택 가능합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 49, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'solar-pro2-250909', 'system_fingerprint': None, 'id': '9924e775-7877-4db9-a08c-82c5f18cf786', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None